# 📊 TikTok Content Analysis
**Dataset:** [Viral Social Media Trends & Engagement Analysis](https://www.kaggle.com/datasets/atharvasoundankar/viral-social-media-trends-and-engagement-analysis) — Atharva Soundankar, Kaggle (CC0 Public Domain)  
**Scope:** TikTok posts filtered from 5 000-row multi-platform dataset  
**Sections:** EDA · Hashtag Analysis · Google Trends · Engagement by Content Type · Posting Day Heatmap · Hashtag Niche Performance · Engagement Level Deep-Dive · Regional Analysis · Correlation Matrix · Key Findings


In [ ]:
# ── Auto-download dataset from Kaggle ─────────────────────────────────────────
# Requires: pip install opendatasets
# You'll be prompted once for your Kaggle username + API key.
# Get your API key at: https://www.kaggle.com/settings → Account → API → Create New Token
import os
import zipfile

DATASET_URL = 'https://www.kaggle.com/datasets/atharvasoundankar/viral-social-media-trends-and-engagement-analysis'
CSV_FILE = 'Cleaned_Viral_Social_Media_Trends.csv'

if os.path.exists(CSV_FILE):
    print(f'✅ {CSV_FILE} already present — skipping download.')
else:
    try:
        import opendatasets as od
    except ImportError:
        import subprocess, sys
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'opendatasets', '-q'])
        import opendatasets as od

    od.download(DATASET_URL, data_dir='.')

    # od downloads into a subfolder — move CSV up
    folder = 'viral-social-media-trends-and-engagement-analysis'
    src = os.path.join(folder, CSV_FILE)
    if os.path.exists(src):
        os.rename(src, CSV_FILE)
        print(f'✅ Downloaded and moved {CSV_FILE}')
    else:
        # list what's in the folder to help debug
        files = os.listdir(folder) if os.path.exists(folder) else []
        print(f'Files in download folder: {files}')
        raise FileNotFoundError(f'{CSV_FILE} not found after download.')


## 1 · Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from wordcloud import WordCloud
import warnings
warnings.filterwarnings('ignore')

# ── Style ──────────────────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 130, 'axes.titlesize': 13, 'axes.labelsize': 11})

# ── Load & filter to TikTok ────────────────────────────────────────────────────
# Download the dataset from Kaggle and place the CSV in the same folder as this notebook.
# File: Cleaned_Viral_Social_Media_Trends.csv
df_all = pd.read_csv('Cleaned_Viral_Social_Media_Trends.csv')
df = df_all[df_all['Platform'] == 'TikTok'].copy()

# ── Parse dates ────────────────────────────────────────────────────────────────
df['Post_Date'] = pd.to_datetime(df['Post_Date'])
df['day_of_week'] = df['Post_Date'].dt.day_name()
df['month'] = df['Post_Date'].dt.strftime('%b %Y')
df['year_month'] = df['Post_Date'].dt.to_period('M')

# ── Derived metrics ────────────────────────────────────────────────────────────
df['engagement_rate'] = (df['Likes'] + df['Comments'] + df['Shares']) / df['Views'].replace(0, np.nan)
df['engagement_rate'] = df['engagement_rate'].clip(upper=1.0)  # cap at 100%

print(f'TikTok posts: {len(df):,}  |  Date range: {df["Post_Date"].min().date()} → {df["Post_Date"].max().date()}')
df.head(3)


## 2 · Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('TikTok Engagement Metric Distributions', fontsize=15, fontweight='bold')

cols  = ['Views', 'Likes', 'Shares', 'Comments']
colors = ['#fe2c55', '#69b3e7', '#80cf6a', '#ffa07a']

for ax, col, color in zip(axes.flat, cols, colors):
    vals = df[col].dropna()
    ax.hist(vals, bins=30, color=color, edgecolor='white', alpha=0.85)
    ax.set_title(col)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M' if x >= 1e6 else f'{x/1e3:.0f}K'))
    ax.set_ylabel('Count')
    median = vals.median()
    ax.axvline(median, color='black', linestyle='--', linewidth=1.2, label=f'Median')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

print(df[cols].describe().applymap(lambda x: f'{x:,.0f}'))


## 3 · Hashtag Frequency & Word Cloud

In [ ]:
tag_counts = df['Hashtag'].value_counts()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
tag_counts.head(12).plot(kind='bar', ax=ax1, color='#fe2c55', edgecolor='white')
ax1.set_title('Top Hashtags on TikTok', fontweight='bold')
ax1.set_xlabel('Hashtag')
ax1.set_ylabel('Post Count')
ax1.tick_params(axis='x', rotation=35)

# Word cloud
wc = WordCloud(
    width=600, height=350,
    background_color='black',
    colormap='RdPink',
    max_words=50
).generate_from_frequencies(tag_counts.to_dict())

ax2.imshow(wc, interpolation='bilinear')
ax2.axis('off')
ax2.set_title('TikTok Hashtag Cloud', fontweight='bold')

plt.tight_layout()
plt.show()

print(f'Unique hashtags: {tag_counts.nunique()}  |  Most common: {tag_counts.index[0]} ({tag_counts.iloc[0]} posts)')


## 4 · Google Trends for TikTok Creator Keywords

In [ ]:
try:
    from pytrends.request import TrendReq
    import time

    pytrends = TrendReq(hl='en-US', tz=0)
    kw_list = ['TikTok', 'TikTok creator', 'viral video', 'content creator']
    pytrends.build_payload(kw_list, cat=0, timeframe='today 12-m', geo='')
    time.sleep(1)
    interest = pytrends.interest_over_time()

    if not interest.empty:
        interest = interest.drop(columns=['isPartial'], errors='ignore')
        interest.plot(figsize=(14, 5), linewidth=2)
        plt.title('Google Search Interest — TikTok Creator Keywords (Last 12 Months)', fontweight='bold')
        plt.ylabel('Search Interest (0–100)')
        plt.xlabel('Date')
        plt.legend(loc='upper left')
        plt.tight_layout()
        plt.show()
    else:
        print('pytrends returned empty data — try again in a few minutes.')

except Exception as e:
    print(f'pytrends unavailable ({e}). Showing static placeholder.')
    kws = ['TikTok', 'TikTok creator', 'viral video', 'content creator']
    x = np.linspace(0, 52, 52)
    fig, ax = plt.subplots(figsize=(14, 5))
    for kw in kws:
        np.random.seed(hash(kw) % 2**32)
        ax.plot(x, 40 + 40 * np.random.rand(52), label=kw, linewidth=2)
    ax.set_title('Google Trends (simulated — install pytrends for live data)', fontweight='bold')
    ax.set_ylabel('Interest')
    ax.legend()
    plt.tight_layout()
    plt.show()


## 5 · Content Type vs Engagement Rate

In [ ]:
ct_stats = df.groupby('Content_Type')['engagement_rate'].agg(['mean', 'median', 'count']).sort_values('mean', ascending=False)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Mean ER by content type
ct_stats['mean'].plot(kind='bar', ax=ax1, color='#fe2c55', edgecolor='white')
ax1.set_title('Mean Engagement Rate by Content Type', fontweight='bold')
ax1.set_ylabel('Mean Engagement Rate')
ax1.set_xlabel('Content Type')
ax1.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=1))
ax1.tick_params(axis='x', rotation=30)

# Box plot
order = ct_stats.index.tolist()
df.boxplot(column='engagement_rate', by='Content_Type', ax=ax2, 
           boxprops=dict(color='#fe2c55'), medianprops=dict(color='black', linewidth=2))
ax2.set_title('Engagement Rate Distribution by Content Type', fontweight='bold')
ax2.set_xlabel('Content Type')
ax2.set_ylabel('Engagement Rate')
plt.suptitle('')
ax2.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()
print(ct_stats.rename(columns={'mean': 'Mean ER', 'median': 'Median ER', 'count': 'Posts'}).to_string())


## 6 · Posting Day × Month Heatmap

In [ ]:
DAY_ORDER = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

heat_df = df.copy()
heat_df['day_of_week'] = pd.Categorical(heat_df['day_of_week'], categories=DAY_ORDER, ordered=True)
heat_df['month_period'] = heat_df['Post_Date'].dt.to_period('M').dt.strftime('%b %Y')

pivot = heat_df.pivot_table(
    values='engagement_rate',
    index='day_of_week',
    columns='month_period',
    aggfunc='median'
).sort_index()

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(
    pivot,
    cmap='RdYlGn',
    annot=True,
    fmt='.2f',
    linewidths=0.4,
    ax=ax,
    cbar_kws={'label': 'Median Engagement Rate'}
)
ax.set_title('Median Engagement Rate: Day of Week × Month', fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Day of Week')
plt.xticks(rotation=40, ha='right')
plt.tight_layout()
plt.show()


## 7 · Hashtag Niche Performance Heatmap

In [ ]:
metrics = ['Views', 'Likes', 'Shares', 'Comments', 'engagement_rate']

# Keep top 10 hashtags by post count
top_tags = df['Hashtag'].value_counts().head(10).index
niche_df = df[df['Hashtag'].isin(top_tags)].copy()

niche_pivot = niche_df.groupby('Hashtag')[metrics].median()

# Z-score normalise for fair colour comparison
niche_norm = (niche_pivot - niche_pivot.mean()) / niche_pivot.std()

fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(
    niche_norm,
    annot=niche_pivot.applymap(lambda x: f'{x/1e6:.1f}M' if x >= 1e6 else f'{x:.2f}' if x < 10 else f'{x/1e3:.0f}K'),
    fmt='s',
    cmap='coolwarm',
    linewidths=0.4,
    ax=ax,
    cbar_kws={'label': 'Z-score (normalised)'}
)
ax.set_title('TikTok Hashtag Niche Performance (Top 10 by Post Count)', fontweight='bold')
ax.set_xlabel('Metric')
ax.set_ylabel('Hashtag')
plt.tight_layout()
plt.show()


## 8 · Engagement Level Deep-Dive

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Distribution
el_counts = df['Engagement_Level'].value_counts()
colors = {'High': '#2ecc71', 'Medium': '#f39c12', 'Low': '#e74c3c'}
el_counts.plot(kind='pie', ax=axes[0],
               colors=[colors.get(l, 'grey') for l in el_counts.index],
               autopct='%1.1f%%', startangle=90, textprops={'fontsize': 11})
axes[0].set_title('Engagement Level Distribution', fontweight='bold')
axes[0].set_ylabel('')

# Mean views by level
order = ['Low', 'Medium', 'High']
el_views = df.groupby('Engagement_Level')['Views'].mean().reindex(order)
el_views.plot(kind='bar', ax=axes[1],
              color=[colors[l] for l in order], edgecolor='white')
axes[1].set_title('Mean Views by Engagement Level', fontweight='bold')
axes[1].set_ylabel('Mean Views')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
axes[1].tick_params(axis='x', rotation=0)

# Content type mix by engagement level
ct_el = df.groupby(['Engagement_Level', 'Content_Type']).size().unstack(fill_value=0)
ct_el = ct_el.reindex(order)
ct_el.div(ct_el.sum(axis=1), axis=0).plot(kind='bar', ax=axes[2], stacked=True, edgecolor='white')
axes[2].set_title('Content Type Mix by Engagement Level', fontweight='bold')
axes[2].set_ylabel('Proportion')
axes[2].legend(loc='upper right', fontsize=7)
axes[2].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()


## 9 · Regional Engagement Analysis

In [ ]:
region_stats = df.groupby('Region').agg(
    Posts=('Views', 'count'),
    Median_Views=('Views', 'median'),
    Median_ER=('engagement_rate', 'median')
).sort_values('Median_Views', ascending=False)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

region_stats['Median_Views'].head(10).plot(kind='bar', ax=ax1, color='#fe2c55', edgecolor='white')
ax1.set_title('Median Views by Region (Top 10)', fontweight='bold')
ax1.set_ylabel('Median Views')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
ax1.tick_params(axis='x', rotation=40)

region_stats['Median_ER'].sort_values(ascending=False).head(10).plot(kind='bar', ax=ax2, color='#69b3e7', edgecolor='white')
ax2.set_title('Median Engagement Rate by Region (Top 10)', fontweight='bold')
ax2.set_ylabel('Median Engagement Rate')
ax2.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=1))
ax2.tick_params(axis='x', rotation=40)

plt.tight_layout()
plt.show()

print(region_stats.to_string())


## 10 · Correlation Matrix

In [ ]:
num_cols = ['Views', 'Likes', 'Shares', 'Comments', 'engagement_rate']
corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(
    corr,
    annot=True,
    fmt='.2f',
    cmap='RdYlGn',
    vmin=-1, vmax=1,
    linewidths=0.5,
    ax=ax,
    square=True
)
ax.set_title('TikTok Engagement Metric Correlation Matrix', fontweight='bold')
plt.tight_layout()
plt.show()


## 11 · Key Findings

In [ ]:
top_tag = df.groupby('Hashtag')['engagement_rate'].median().idxmax()
top_tag_er = df.groupby('Hashtag')['engagement_rate'].median().max()
top_ct = df.groupby('Content_Type')['engagement_rate'].median().idxmax()
top_region = df.groupby('Region')['Views'].median().idxmax()
best_day = df.groupby('day_of_week')['engagement_rate'].median().idxmax()
high_pct = (df['Engagement_Level'] == 'High').mean() * 100

findings = [
    f'📌 Dataset: {len(df):,} TikTok posts ({df["Post_Date"].min().date()} – {df["Post_Date"].max().date()})',
    f'🏷️  Highest ER hashtag: {top_tag} ({top_tag_er:.1%} median engagement rate)',
    f'🎬  Best performing content type: {top_ct}',
    f'🌍  Highest median views by region: {top_region}',
    f'📅  Best posting day: {best_day} (highest median ER)',
    f'🔥  {high_pct:.1f}% of TikTok posts classified as High engagement',
    f'📊  Median views: {df["Views"].median():,.0f} | Median likes: {df["Likes"].median():,.0f}',
]

print('\n'.join(findings))
